# 04 — Simulate change

Re-generate briefs for the worst papers using the proposed template, re-judge them, and compare scores.

**Input:** `03-proposed-change.md`, `01-worst-briefs.jsonl`, corpus `.txt` files, `corpus/manifest.jsonl` (bibliographic metadata).

**Output:** `04-simulation.jsonl` — per-paper before/after scores and deltas.

See [paper-brief-improvement.md](../../docs/specs/paper-brief-improvement.md) step 4.

In [1]:
# Chat model id for brief generation.
# Use the API model name, not the run folder slug. Example: "gemma4:e4b".
# Leave empty to infer from RUN_ID.
MODEL = ""

# Judge model. Leave empty to use MODEL.
JUDGE_MODEL = ""

# Improvement run folder under data/paper_brief_improvement/.
# Leave empty to use the latest folder that has 03-proposed-change.md.
RUN_ID = ""

In [2]:
from __future__ import annotations

import json
import os
import re
import statistics
from pathlib import Path
from unittest.mock import patch

from IPython.display import Markdown, display

from paper_reviewer.schemas.topic_scope.generate_paper_brief import PaperBriefContent
from paper_reviewer.schemas.topic_scope.paper_brief_evaluation import (
    mean_evaluation_score,
)
from paper_reviewer.topic_scope.generate_paper_brief.llm import (
    generate_paper_brief_content,
    load_paper_brief_template,
)
from paper_reviewer.topic_scope.paper_brief_evaluation.llm import (
    judge_paper_brief_evaluation,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
IMPROVEMENT_PARENT = REPO_ROOT / "data" / "paper_brief_improvement"
EVAL_PARENT = REPO_ROOT / "data" / "paper_brief_evaluation"
CORPUS_DIR = EVAL_PARENT / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.jsonl"

print(f"repo root: {REPO_ROOT}")
print(f"improvement parent: {IMPROVEMENT_PARENT}")
print(f"manifest: {MANIFEST_PATH}")

repo root: /workspace
improvement parent: /workspace/data/paper_brief_improvement
manifest: /workspace/data/paper_brief_evaluation/corpus/manifest.jsonl


In [3]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")
CRITERIA = ("faithfulness", "completeness", "conciseness", "topic_agnostic")
_CHANGE_PAIR_PATTERN = re.compile(
    r"\*\*Old Text:\*\*\s*\n```(?:markdown)?\n(.*?)```\s*\n\*\*New Text:\*\*\s*\n```(?:markdown)?\n(.*?)```",
    re.DOTALL,
)


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def resolve_run_dir(run_id: str) -> Path:
    if run_id:
        d = IMPROVEMENT_PARENT / run_id
        if not (d / "03-proposed-change.md").is_file():
            raise FileNotFoundError(f"No 03-proposed-change.md in {d}")
        if not (d / "01-worst-briefs.jsonl").is_file():
            raise FileNotFoundError(f"No 01-worst-briefs.jsonl in {d}")
        return d
    candidates = sorted(
        (
            p.parent
            for p in IMPROVEMENT_PARENT.glob("*/03-proposed-change.md")
            if _RUN_ID_PATTERN.match(p.parent.name)
        ),
        key=lambda d: d.name,
    )
    if not candidates:
        raise FileNotFoundError(
            "No improvement runs with 03-proposed-change.md found under "
            + str(IMPROVEMENT_PARENT)
        )
    return candidates[-1]


def model_slug_from_run_id(run_id: str) -> str | None:
    if not _RUN_ID_PATTERN.match(run_id):
        return None
    _, _, slug = run_id.partition("_")
    return slug or None


def chat_model_from_run_slug(slug: str) -> str:
    if ":" in slug:
        return slug
    if "-" not in slug:
        return slug
    return slug.replace("-", ":", 1)


def resolve_chat_model(model: str, run_id: str) -> str:
    slug = model_slug_from_run_id(run_id)
    candidate = model.strip()
    if not candidate and slug:
        inferred = chat_model_from_run_slug(slug)
        print(f"MODEL empty; inferred {inferred!r} from run_id slug {slug!r}")
        return inferred
    if not candidate:
        raise ValueError(
            "MODEL is required. Set the chat model id (example: 'gemma4:e4b')."
        )
    if slug and candidate == slug and ":" not in candidate:
        inferred = chat_model_from_run_slug(slug)
        print(
            f"MODEL looks like a run_id slug ({candidate!r}); "
            f"using chat model id {inferred!r} instead"
        )
        return inferred
    return candidate


def parse_proposed_changes(proposal_md: str) -> list[tuple[str, str]]:
    pairs = [
        (old.strip(), new.strip())
        for old, new in _CHANGE_PAIR_PATTERN.findall(proposal_md)
    ]
    if not pairs:
        raise ValueError(
            "No **Old Text:** / **New Text:** pairs found in 03-proposed-change.md"
        )
    return pairs


def apply_proposed_changes(template: str, pairs: list[tuple[str, str]]) -> str:
    result = template
    for index, (old, new) in enumerate(pairs, start=1):
        if old not in result:
            raise ValueError(
                f"Change {index}: old text not found in current template:\n{old[:300]}"
            )
        result = result.replace(old, new, 1)
    return result


def manifest_by_doi(path: Path) -> dict[str, dict]:
    by_doi: dict[str, dict] = {}
    for row in load_jsonl(path):
        doi = row.get("doi")
        if isinstance(doi, str) and doi:
            by_doi[doi] = row
    return by_doi


def criterion_score(evaluation: dict, name: str) -> int | None:
    entry = evaluation.get(name, {})
    if not isinstance(entry, dict):
        return None
    score = entry.get("score")
    if isinstance(score, bool) or not isinstance(score, int):
        return None
    return score

In [4]:
run_dir = resolve_run_dir(RUN_ID)
run_id = run_dir.name
generator_model = resolve_chat_model(MODEL, run_id)
judge_model = JUDGE_MODEL.strip() or generator_model
print(f"generator model: {generator_model}")
print(f"judge model: {judge_model}")
print(f"run dir: {run_dir.relative_to(REPO_ROOT)}")

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing corpus manifest: {MANIFEST_PATH}")

worst_rows = load_jsonl(run_dir / "01-worst-briefs.jsonl")
proposal_md = (run_dir / "03-proposed-change.md").read_text(encoding="utf-8")
change_pairs = parse_proposed_changes(proposal_md)
current_template = load_paper_brief_template()
proposed_template = apply_proposed_changes(current_template, change_pairs)
manifest = manifest_by_doi(MANIFEST_PATH)

print(f"worst briefs: {len(worst_rows)}")
print(f"template changes: {len(change_pairs)}")
print(f"proposed template length: {len(proposed_template)} chars")

MODEL empty; inferred 'gemma4:e4b' from run_id slug 'gemma4-e4b'
generator model: gemma4:e4b
judge model: gemma4:e4b
run dir: data/paper_brief_improvement/20260818T221210Z_gemma4-e4b
worst briefs: 5
template changes: 1
proposed template length: 3874 chars


In [5]:
simulation_rows: list[dict] = []

for i, row in enumerate(worst_rows):
    doi = row["doi"]
    original_score = float(row["evaluation_score"])
    corpus_path = REPO_ROOT / row["corpus_file"]
    meta = manifest.get(doi)

    print(f"[{i+1}/{len(worst_rows)}] {doi} (original={original_score:.2f}) ...", end=" ", flush=True)

    if not corpus_path.is_file():
        print("SKIP corpus missing")
        simulation_rows.append({
            "doi": doi,
            "original_score": original_score,
            "error": "corpus file missing",
        })
        continue
    if meta is None or not meta.get("title"):
        print("SKIP manifest missing title")
        simulation_rows.append({
            "doi": doi,
            "original_score": original_score,
            "error": "manifest row missing for doi",
        })
        continue

    full_text = corpus_path.read_text(encoding="utf-8")
    try:
        os.environ["OPENAI_MODEL"] = generator_model
        with patch(
            "paper_reviewer.topic_scope.generate_paper_brief.llm.load_paper_brief_template",
            return_value=proposed_template,
        ):
            brief_result = generate_paper_brief_content(
                full_text,
                title=str(meta["title"]),
                journal=meta.get("journal"),
                published_year=meta.get("published_year"),
            )

        os.environ["OPENAI_MODEL"] = judge_model
        evaluation = judge_paper_brief_evaluation(
            full_text,
            content=brief_result.content,
        )
        new_score = float(mean_evaluation_score(evaluation))
        delta = round(new_score - original_score, 2)
        simulation_rows.append({
            "doi": doi,
            "original_score": original_score,
            "new_score": new_score,
            "delta": delta,
            "new_evaluation": evaluation.model_dump(mode="json"),
            "new_brief": brief_result.content.model_dump(mode="json"),
        })
        print(f"new={new_score:.2f} delta={delta:+.2f}")
    except Exception as exc:
        print(f"ERROR: {exc}")
        simulation_rows.append({
            "doi": doi,
            "original_score": original_score,
            "error": str(exc),
        })

success_count = sum(1 for r in simulation_rows if "new_score" in r)
print(f"\nsimulated: {success_count} / {len(simulation_rows)}")

[1/5] 10.1093/JME/TJAG095 (original=4.00) ... new=5.00 delta=+1.00
[2/5] 10.3390/VACCINES14060499 (original=4.00) ... new=5.00 delta=+1.00
[3/5] 10.64898/2026.05.10.722846 (original=4.00) ... new=5.00 delta=+1.00
[4/5] 10.1016/J.APSB.2026.02.021 (original=4.25) ... new=4.00 delta=-0.25
[5/5] 10.1038/S41467-026-73251-5 (original=4.25) ... new=4.25 delta=+0.00

simulated: 5 / 5


In [6]:
sim_path = run_dir / "04-simulation.jsonl"
with sim_path.open("w", encoding="utf-8") as fh:
    for row in simulation_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {len(simulation_rows)} rows to {sim_path.relative_to(REPO_ROOT)}")

wrote 5 rows to data/paper_brief_improvement/20260818T221210Z_gemma4-e4b/04-simulation.jsonl


In [7]:
successful = [r for r in simulation_rows if "new_score" in r]
if not successful:
    raise RuntimeError(
        "No successful simulations. Check errors in the loop output above."
    )

deltas = [r["delta"] for r in successful]
improved = sum(1 for d in deltas if d > 0)
degraded = sum(1 for d in deltas if d < 0)
unchanged = sum(1 for d in deltas if d == 0)

lines = [
    "### Simulation summary",
    "",
    f"- Papers simulated: {len(successful)}",
    f"- Mean delta: {statistics.mean(deltas):+.2f}",
    f"- Median delta: {statistics.median(deltas):+.2f}",
    f"- Improved: {improved}",
    f"- Degraded: {degraded}",
    f"- Unchanged: {unchanged}",
    "",
    "| DOI | Original | New | Delta | " + " | ".join(CRITERIA) + " |",
    "| --- | ---: | ---: | ---: | " + " | ".join(["---:"] * len(CRITERIA)) + " |",
]

criterion_deltas: dict[str, list[float]] = {c: [] for c in CRITERIA}
worst_by_doi = {r["doi"]: r for r in worst_rows}

for row in successful:
    doi = row["doi"]
    original_eval = worst_by_doi[doi]["evaluation"]
    new_eval = row["new_evaluation"]
    criterion_cells = []
    for c in CRITERIA:
        old_s = criterion_score(original_eval, c)
        new_s = criterion_score(new_eval, c)
        if old_s is not None and new_s is not None:
            delta_c = new_s - old_s
            criterion_deltas[c].append(float(delta_c))
            criterion_cells.append(f"{new_s} ({delta_c:+d})")
        else:
            criterion_cells.append("–")
    lines.append(
        f"| `{doi}` | {row['original_score']:.2f} | {row['new_score']:.2f} | "
        f"{row['delta']:+.2f} | " + " | ".join(criterion_cells) + " |"
    )

lines.extend(["", "### Mean criterion delta (new − original)"])
for c in CRITERIA:
    values = criterion_deltas[c]
    if values:
        lines.append(f"- **{c}**: {statistics.mean(values):+.2f}")
    else:
        lines.append(f"- **{c}**: –")

summary_md = "\n".join(lines)
display(Markdown(summary_md))

### Simulation summary

- Papers simulated: 5
- Mean delta: +0.55
- Median delta: +1.00
- Improved: 3
- Degraded: 1
- Unchanged: 1

| DOI | Original | New | Delta | faithfulness | completeness | conciseness | topic_agnostic |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `10.1093/JME/TJAG095` | 4.00 | 5.00 | +1.00 | 5 (+3) | 5 (+0) | 5 (+1) | 5 (+0) |
| `10.3390/VACCINES14060499` | 4.00 | 5.00 | +1.00 | 5 (+3) | 5 (+0) | 5 (+1) | 5 (+0) |
| `10.64898/2026.05.10.722846` | 4.00 | 5.00 | +1.00 | 5 (+3) | 5 (+0) | 5 (+1) | 5 (+0) |
| `10.1016/J.APSB.2026.02.021` | 4.25 | 4.00 | -0.25 | 3 (+0) | 4 (-1) | 4 (+0) | 5 (+0) |
| `10.1038/S41467-026-73251-5` | 4.25 | 4.25 | +0.00 | 5 (+0) | 3 (-1) | 4 (-1) | 5 (+2) |

### Mean criterion delta (new − original)
- **faithfulness**: +1.80
- **completeness**: -0.40
- **conciseness**: +0.40
- **topic_agnostic**: +0.40